In [2]:
!pip install playwright
!playwright install chromium  # installe le navigateur Chromium

(node:2155) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
162.3 MiB [                    ] 0% 278.1s162.3 MiB [                    ] 0% 620.5s162.3 MiB [                    ] 0% 538.7s162.3 MiB [                    ] 0% 537.2s162.3 MiB [                    ] 0% 406.2s162.3 MiB [                    ] 0% 363.3s162.3 MiB [                    ] 0% 374.3s162.3 MiB [                    ] 0% 408.7s162.3 MiB [                    ] 0% 370.0s162.3 MiB [                    ] 0% 367.8s162.3 MiB [                    ] 0% 300.3s162.3 MiB [                    ] 0% 310.1s162.3 MiB [                    ] 0% 296.7s162.3 MiB [                    ] 0% 281.5s162.3 MiB [                    ] 0% 270.2s162.3 MiB [                    ] 0% 250.4s162.3 MiB [                 

In [17]:
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.google.co.ma/travel/hotels/entity/CgoIldD76sPvxI8cEAE/reviews?utm_campaign=sharing&utm_medium=link&utm_source=htls&ts=CAEaIAoCGgASGhIUCgcI6g8QBRgOEgcI6g8QBRgPGAEyAhAAKgkKBToDTUFEGgA&ved=0CAAQ5JsGahcKEwiguKOssYmUAxUAAAAAHQAAAAAQBA"
HOTEL_NAME = "Hotel Oujda"
HOTEL_CITY = "Oujda"
MAX_SCROLLS = 200
# ==================================================


def human_sleep(a=1.0, b=2.5):
    time.sleep(random.uniform(a, b))


class GoogleHotelsScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=fr-FR')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(120)
        self.wait = WebDriverWait(self.driver, 30)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _t(self, parent, css, idx=0):
        try:
            els = parent.find_elements(By.CSS_SELECTOR, css)
            return els[idx].text.strip() if len(els) > idx else ""
        except Exception:
            return ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _click_google_filter(self):
        print("   🔍 Activation filtre Google...")
        human_sleep(2, 3)

        for xpath in [
            "//span[text()='Tous les avis']/ancestor::div[@role='option']",
            "//div[@aria-selected='true'][@role='option']",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                btn.click()
                print("   ✅ Dropdown ouvert")
                human_sleep(2, 3)
                break
            except Exception:
                pass

        for xpath in [
            "//div[@role='option'][@aria-label='Google']",
            "//span[text()='Google']/ancestor::div[@role='option']",
        ]:
            try:
                google_btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].click();", google_btn)
                print("   ✅ Filtre Google activé")
                human_sleep(3, 5)
                return True
            except Exception:
                pass
        return False

    def _scroll_with_keyboard(self):
        """
        ✅ Scroller avec les touches clavier sur la dernière carte visible
        — contourne le problème de conteneur non trouvé
        """
        try:
            cards = self.driver.find_elements(By.CSS_SELECTOR, "div.Svr5cf.bKhjM")
            if cards:
                last_card = cards[-1]
                # Cliquer sur la dernière carte pour focus
                ActionChains(self.driver)\
                    .move_to_element(last_card)\
                    .click()\
                    .perform()
                time.sleep(0.3)
                # Appuyer sur END ou PAGE_DOWN plusieurs fois
                for _ in range(5):
                    ActionChains(self.driver)\
                        .send_keys(Keys.PAGE_DOWN)\
                        .perform()
                    time.sleep(0.2)
                return True
        except Exception:
            pass
        return False

    def _scroll_last_card_into_view(self):
        """
        ✅ Scroller la dernière carte dans la vue
        — force Google à charger les suivantes
        """
        try:
            cards = self.driver.find_elements(By.CSS_SELECTOR, "div.Svr5cf.bKhjM")
            if cards:
                last = cards[-1]
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({behavior: 'smooth', block: 'end'});",
                    last
                )
                return True
        except Exception:
            pass
        return False

    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'Plus') or contains(text(),'More')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script(
                        "arguments[0].scrollIntoView({block:'center'});", btn
                    )
                    btn.click()
                    time.sleep(0.2)
                except Exception:
                    pass
        except Exception:
            pass

    def _parse_review(self, card):
        name = ""
        for sel in ["a.DHIhE.QB2Jof", "a[class*='DHIhE']"]:
            name = self._t(card, sel)
            if name:
                break

        rating_raw = self._t(card, "div.GDWaad")
        rating = rating_raw.split("/")[0].strip() if "/" in rating_raw else rating_raw

        date = ""
        for sel in ["span.iUtr1.CQYfx", "span[class*='iUtr1']"]:
            date = self._t(card, sel)
            if date:
                break

        trip_type = self._t(card, "div.ThUm5b span")

        text = ""
        for sel in ["div.K7oBsc span", "div[class*='K7oBsc'] span", "div.STQFb span"]:
            text = self._t(card, sel)
            if text:
                break

        source = "google.com"
        try:
            src_els = card.find_elements(By.CSS_SELECTOR, "span.OedQsc.FnxmJb")
            if src_els:
                source = src_els[0].text.strip().lower()
        except Exception:
            pass

        return {
            "source": source,
            "hotel_name": HOTEL_NAME,
            "hotel_city": HOTEL_CITY,
            "reviewer_name": name,
            "rating": rating,
            "review_text": text,
            "trip_type": trip_type,
            "publication_date": date,
            "language": self.detect_language(text),
        }

    def scrape(self, url, max_scrolls=200):
        print(f"🔄 Scraping Google Hotels : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(5, 8)

        # Cookie
        for sel in ["button[id='L2AGLb']", "button[id='W0wltc']",
                    "button[aria-label='Tout accepter']"]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(2, 3)
                print("   🔕 Cookie accepté")
                break
            except Exception:
                pass

        # Filtre Google
        self._click_google_filter()

        all_reviews = []
        seen = set()
        scroll_count = 0
        no_new_count = 0

        print("   📜 Collecte des avis Google...")

        while scroll_count < max_scrolls:
            self._expand_reviews()

            cards = self.driver.find_elements(By.CSS_SELECTOR, "div.Svr5cf.bKhjM")

            new_count = 0
            for card in cards:
                review = self._parse_review(card)
                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"   📄 Scroll {scroll_count+1} | +{new_count} | Total : {len(all_reviews)}")

            if new_count == 0:
                no_new_count += 1
                if no_new_count >= 5:
                    print("   ✅ Fin du scroll.")
                    break
            else:
                no_new_count = 0

            # ✅ Triple stratégie de scroll
            self._scroll_last_card_into_view()
            human_sleep(0.5, 1)
            self._scroll_with_keyboard()
            human_sleep(0.5, 1)
            # Scroll JS sur body en dernier recours
            self.driver.execute_script(
                "document.body.scrollTop += 1000;"
                "document.documentElement.scrollTop += 1000;"
            )
            human_sleep(2.0, 3.5)
            scroll_count += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = GoogleHotelsScraper()
    df = scraper.scrape(URL, max_scrolls=MAX_SCROLLS)

    if not df.empty:
        filename = f"avis_Google_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "publication_date", "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Sources présentes :")
        if "source" in df.columns:
            print(df["source"].value_counts())
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping Google Hotels : Hotel Oujda
   🔍 Activation filtre Google...
   ✅ Dropdown ouvert
   ✅ Filtre Google activé
   📜 Collecte des avis Google...
   📄 Scroll 1 | +10 | Total : 10
   📄 Scroll 2 | +20 | Total : 30
   📄 Scroll 3 | +20 | Total : 50
   📄 Scroll 4 | +30 | Total : 80
   📄 Scroll 5 | +20 | Total : 100
   📄 Scroll 6 | +20 | Total : 120
   📄 Scroll 7 | +30 | Total : 150
   📄 Scroll 8 | +10 | Total : 160
   📄 Scroll 9 | +20 | Total : 180
   📄 Scroll 10 | +30 | Total : 210
   📄 Scroll 11 | +10 | Total : 220
   📄 Scroll 12 | +30 | Total : 250
   📄 Scroll 13 | +20 | Total : 270
   📄 Scroll 14 | +30 | Total : 300
   📄 Scroll 15 | +20 | Total : 320
   📄 Scroll 16 | +20 | Total : 340
   📄 Scroll 17 | +20 | Total : 360
   📄 Scroll 18 | +30 | Total : 390
   📄 Scroll 19 | +20 | Total : 410
   📄 Scroll 20 | +20 | Total : 430
   📄 Scroll 21 | +20 | Total : 450
   📄 Scroll 22 | +20 | Total : 470
   📄 Scroll 23 | +20 | Total : 490
   📄 Scroll 24 | +20 | Total : 510
   📄 Scroll 25 | +10 

In [18]:
df.head()

,source,hotel_name,hotel_city,reviewer_name,rating,review_text,trip_type,publication_date,language
0,google.com,Hotel Oujda,Oujda,Mohamed Salmane KAMAL,5,Chambre très propre et calme personnel accueil...,Couple,il y a 4 mois sur Google,french
1,google.com,Hotel Oujda,Oujda,Slimane Galaxy,5,Belle experience avec mes amis.\nLa propreté e...,Amis,il y a 4 mois sur Google,french
2,google.com,Hotel Oujda,Oujda,Bahae Hadri,5,Mon séjour a été particulièrement plaisant. J’...,Vacances ❘ Couple,il y a 3 mois sur Google,french
3,google.com,Hotel Oujda,Oujda,Dimi,2,"Chambres bien , propres.\nPersonnels plutôt sy...",,il y a un mois sur Google,french
4,google.com,Hotel Oujda,Oujda,CENDRINE TMIMI,5,Chambre fonctionnelle et agréable.\nEntretien ...,,il y a 4 mois sur Google,french


In [4]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Rabat-Hotels-Ibis-Rabat-Agdal.h17038191.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Rabat Agdal"
HOTEL_CITY = "Rabat"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Rabat Agdal
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   16 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   16 avis visibles — on continue !
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +10 | Total : 80
      'More reviews' clique
   Click 8 | +10 | Total : 90
      'More reviews' clique
   Click 9 | +10 | Total : 100
      'More reviews' clique
   Click 10 | +10 | Total : 110
      'More reviews' clique
   Click 11 | +10 | Total : 120
      'More reviews' clique
   Click 12 | +10 | Total : 130
      'More reviews' clique
   Click 13 | +10 | Total : 140
      'More reviews' clique
   Click 14 | +10 | Total : 150
      'More reviews' clique
   Click 15 | +

In [3]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Casablanca-Hotels-Ibis-Casa-Voyageurs.h83602.Hotel-Information?challengeReferer=www.google.com&pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Casa Voyageur"
HOTEL_CITY = "Casablanca"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Casa Voyageur
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   Attente... (1/10)
   Attente... (2/10)
   Attente... (3/10)
   Attente... (4/10)
   Attente... (5/10)
   18 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +9 | Total : 79
      'More reviews' clique
   Click 8 | +10 | Total : 89
      'More reviews' clique
   Click 9 | +10 | Total : 99
      'More reviews' clique
   Click 10 | +10 | Total : 109
      'More reviews' clique
   Click 11 | +10 | Total : 119
      'More reviews' clique
   Click 12 | +10 | Total 

In [2]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Casablanca-Hotels-Ibis-Casablanca-Nearshore.h7901044.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Casablanca Nearshore"
HOTEL_CITY = "Casablanca"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Casablanca Nearshore
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   11 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +10 | Total : 80
      'More reviews' clique
   Click 8 | +10 | Total : 90
      'More reviews' clique
   Click 9 | +10 | Total : 100
      'More reviews' clique
   Click 10 | +10 | Total : 110
      'More reviews' clique
   Click 11 | +10 | Total : 120
      'More reviews' clique
   Click 12 | +10 | Total : 130
      'More reviews' clique
   Click 13 | +10 | Total : 140
      'More reviews' clique
   Click 

In [19]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Marrakech-Hotels-Ibis-Marrakech-Palmeraie-Hotel.h1568732.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Marrakech Palmeraie"
HOTEL_CITY = "Marrakech"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Marrakech Palmeraie
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   10 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +10 | Total : 80
      'More reviews' clique
   Click 8 | +10 | Total : 90
      'More reviews' clique
   Click 9 | +10 | Total : 100
      'More reviews' clique
   Click 10 | +10 | Total : 110
      'More reviews' clique
   Click 11 | +10 | Total : 120
      'More reviews' clique
   Click 12 | +10 | Total : 130
      'More reviews' clique
   Click 13 | +10 | Total : 140
      'More reviews' clique
   Click 

In [22]:
df.head()

,source,hotel_name,hotel_city,reviewer_name,rating,review_text,trip_type,publication_date,stay_date,language
0,expedia.com,Ibis Marrakech Palmeraie,Marrakech,Khaoula,8,Great,"Mar 8, 2026",,2 nights in Feb 2026,english
1,expedia.com,Ibis Marrakech Palmeraie,Marrakech,Zain,10,Clean room professional staff and nice breakfast,"Business traveler, traveled with family, trave...","Dec 4, 2025",4 nights in Nov 2025,english
2,expedia.com,Ibis Marrakech Palmeraie,Marrakech,Gizaw Wolde,8,The room lacks TV and good caviars.,Solo traveler,"Feb 24, 2026",8 nights in Feb 2026,english
3,expedia.com,Ibis Marrakech Palmeraie,Marrakech,Carla,8,Very nice check-in at 2am after out flight was...,Traveled with family,"Nov 1, 2025",1 night in Oct 2025,english
4,expedia.com,Ibis Marrakech Palmeraie,Marrakech,Sodiq,8,"Convenient, and very walkable area","Sep 29, 2025",,6 nights in Sep 2025,english


In [18]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Ouarzazate-Hotels-Ibis-Ouarzazate-Centre.h1377929.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Ouarzazate"
HOTEL_CITY = "Ouarzazate"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Ouarzazate
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   20 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +10 | Total : 80
      'More reviews' clique
   Click 8 | +10 | Total : 90
      'More reviews' clique
   Click 9 | +10 | Total : 100
      'More reviews' clique
   Click 10 | +10 | Total : 110
      'More reviews' clique
   Click 11 | +10 | Total : 120
      'More reviews' clique
   Click 12 | +10 | Total : 130
      'More reviews' clique
   Click 13 | +10 | Total : 140
      'More reviews' clique
   Click 

In [17]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/El-Jadida-Hotels-Hotel-Ibis-El-Jadida.h1218076.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis El jadida"
HOTEL_CITY = "El jadida"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis El jadida
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   19 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +23 | Total : 43
      'More reviews' clique
   Click 3 | +10 | Total : 53
      'More reviews' clique
   Click 4 | +8 | Total : 61
      'More reviews' clique
   Click 5 | +0 | Total : 61
      'More reviews' clique
   Click 6 | +10 | Total : 71
      'More reviews' clique
   Click 7 | +10 | Total : 81
      'More reviews' clique
   Click 8 | +10 | Total : 91
      'More reviews' clique
   Click 9 | +10 | Total : 101
      'More reviews' clique
   Click 10 | +10 | Total : 111
      'More reviews' clique
   Click 11 | +10 | Total : 121
      'More reviews' clique
   Click 12 | +10 | Total : 131
      'More reviews' clique
   Click 13 | +20 | Total : 151
      'More reviews' clique
   Click 14

In [16]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Marrakech-Hotels-Hotel-Ibis-Marrakech-Centre-Gare.h564686.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Marrakech centre gare"
HOTEL_CITY = "Marrakech"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Marrakech centre gare
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   12 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +0 | Total : 70
      'More reviews' clique
   Click 8 | +0 | Total : 70
      'More reviews' clique
   Click 9 | +0 | Total : 70
      'More reviews' clique
   Click 10 | +10 | Total : 80
      'More reviews' clique
   Click 11 | +10 | Total : 90
      'More reviews' clique
   Click 12 | +10 | Total : 100
      'More reviews' clique
   Click 13 | +10 | Total : 110
      'More reviews' clique
   Click 14 | +

In [15]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Meknes-Hotels-Ibis-Meknes-Hotel.h903853.Hotel-Information?challengeReferer=www.google.com&pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Meknes"
HOTEL_CITY = "Meknes"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Meknes
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   18 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +20 | Total : 90
      'More reviews' clique
   Click 8 | +10 | Total : 100
      'More reviews' clique
   Click 9 | +0 | Total : 100
      'More reviews' clique
   Click 10 | +0 | Total : 100
      'More reviews' clique
   Click 11 | +0 | Total : 100
      'More reviews' clique
   Click 12 | +0 | Total : 100
      'More reviews' clique
   Click 13 | +0 | Total : 100
      'More reviews' clique
   Click 14 |

In [14]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Fes-Hotels-Hotel-Ibis-Fes.h564677.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Fes"
HOTEL_CITY = "Fes"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Fes
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   14 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +10 | Total : 80
      'More reviews' clique
   Click 8 | +10 | Total : 90
      'More reviews' clique
   Click 9 | +10 | Total : 100
      'More reviews' clique
   Click 10 | +10 | Total : 110
      'More reviews' clique
   Click 11 | +10 | Total : 120
      'More reviews' clique
   Click 12 | +9 | Total : 129
      'More reviews' clique
   Click 13 | +10 | Total : 139
      'More reviews' clique
   Click 1

In [9]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Oujda-Hotels-Hotel-Ibis-Oujda.h564684.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Oujda"
HOTEL_CITY = "Oujda"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Oujda
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   19 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +0 | Total : 10
      'More reviews' clique
   Click 2 | +10 | Total : 20
      'More reviews' clique
   Click 3 | +10 | Total : 30
      'More reviews' clique
   Click 4 | +10 | Total : 40
      'More reviews' clique
   Click 5 | +10 | Total : 50
      'More reviews' clique
   Click 6 | +10 | Total : 60
      'More reviews' clique
   Click 7 | +10 | Total : 70
      'More reviews' clique
   Click 8 | +10 | Total : 80
      'More reviews' clique
   Click 9 | +10 | Total : 90
      'More reviews' clique
   Click 10 | +8 | Total : 98
      'More reviews' clique
   Click 11 | +27 | Total : 125
      'More reviews' clique
   Click 12 | +4 | Total : 129
      Retry — scroll complet + attente 2s...
      Bouton 'More reviews' introuvable — fin des avis.
 

In [5]:
df.head()

,source,hotel_name,hotel_city,reviewer_name,rating,review_text,trip_type,publication_date,stay_date,language
0,expedia.com,Hotel Ibis Tanger City Center,Tanger,What guests liked,,,,,,unknown
1,expedia.com,Hotel Ibis Tanger City Center,Tanger,MOHEB Mohsen,All reviews,good for one or two days trip,Traveled with partner,"Feb 20, 2026",2 nights in Feb 2026,english
2,expedia.com,Hotel Ibis Tanger City Center,Tanger,MOHEB Mohsen,,Traveled with partner,Traveled with partner,"Feb 20, 2026",,english
3,expedia.com,Hotel Ibis Tanger City Center,Tanger,Ratawan,,Traveled with partner,Traveled with partner,"Mar 8, 2026",,english
4,expedia.com,Hotel Ibis Tanger City Center,Tanger,Ryan,,"Dec 25, 2025","Dec 25, 2025",,,unknown


In [8]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Casablanca-Hotels-Hotel-Ibis-Casa-Sidi-Maarouf.h1604264.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Casa Sidi Maarouf"
HOTEL_CITY = "Casablanca"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Casa Sidi Maarouf
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   Attente... (1/10)
   Attente... (2/10)
   Attente... (3/10)
   Attente... (4/10)
   Attente... (5/10)
   11 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +10 | Total : 80
      'More reviews' clique
   Click 8 | +10 | Total : 90
      'More reviews' clique
   Click 9 | +10 | Total : 100
      'More reviews' clique
   Click 10 | +10 | Total : 110
      'More reviews' clique
   Click 11 | +10 | Total : 120
      'More reviews' clique
   Click 12 | +10 | Tota

In [10]:
df.head()

,source,hotel_name,hotel_city,reviewer_name,rating,review_text,trip_type,publication_date,stay_date,language
0,expedia.com,Hotel Ibis Tanger City Center,Tanger,,,Hotel near Tanger City Center,,,,english
1,expedia.com,Hotel Ibis Tanger City Center,Tanger,MOHEB Mohsen,,Traveled with partner,Traveled with partner,"Feb 20, 2026",,english
2,expedia.com,Hotel Ibis Tanger City Center,Tanger,Ratawan,,Traveled with partner,Traveled with partner,"Mar 8, 2026",,english
3,expedia.com,Hotel Ibis Tanger City Center,Tanger,Ryan,,"Dec 25, 2025","Dec 25, 2025",,,unknown
4,expedia.com,Hotel Ibis Tanger City Center,Tanger,Hector,,Solo traveler,Solo traveler,"Jan 5, 2026",,unknown


In [5]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Casablanca-Hotels-Ibis-Casablanca-City-Center.h2871181.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis Casablanca City Center"
HOTEL_CITY = "Casablanca"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis Casablanca City Center
   Popup ferme
   Total avis detecte : 769  (depuis 'See all 769 reviews')
   max_clicks calcule : ceil(769/10) + 10 = 87
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton clique : 'see all 769 reviews'
   Bouton clique : 'see all 769 reviews'
   Bouton reviews clique (XPath)

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   10 avis visibles — on continue !
   Collecte des avis (max 87 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +10 | Total : 80
      'More reviews' clique
   Click 8 | +10 | Total : 90
      'More reviews' clique
   Click 9 | +10 | Total : 100
      'More reviews' clique
   Click 10 | +10 | Total : 110
      'More reviews' clique
   Click 11 | +10 | Total : 120
      'More reviews' clique
   Click 12 | +10 | Total : 130
      'More reviews' clique
   Click 13 | +10 | Total : 140
      'More reviews' clique
   Click 14 | +10 | Total : 150
      'More reviews' clique
   Click 15 | +9

In [6]:
df.head()

,source,hotel_name,hotel_city,reviewer_name,rating,review_text,trip_type,publication_date,stay_date,language
0,expedia.com,Ibis Casablanca City Center,Casablanca,Juanita,10,Great location. Near the airport bus stop and ...,Solo traveler,"Apr 10, 2026",1 night in Apr 2026,english
1,expedia.com,Ibis Casablanca City Center,Casablanca,Deborah,8,Perfect hotel for the budget. Breakfast was ge...,Solo traveler,"Mar 18, 2026",1 night in Mar 2026,english
2,expedia.com,Ibis Casablanca City Center,Casablanca,Oxana,10,The rooms are fairly standard in size and furn...,Traveled with group,"Apr 5, 2026",1 night in Apr 2026,english
3,expedia.com,Ibis Casablanca City Center,Casablanca,Alexis,10,Spotless property and the room was spacious an...,Traveled with family,"Feb 4, 2026",1 night in Jan 2026,english
4,expedia.com,Ibis Casablanca City Center,Casablanca,Barry,8,Very friendly and helpful staff and very clean...,Traveled with family,"Dec 19, 2025",1 night in Dec 2025,english


In [4]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Tangier-Hotels-Ibis-Tanger-City-Center.h4177427.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Hotel Ibis Tanger City Center"
HOTEL_CITY = "Tanger"
REVIEWS_PER_CLICK = 10   # Expedia charge ~10 avis par clic "More reviews"
SAFETY_MARGIN     = 10   # clics supplementaires pour ne pas manquer les derniers
# MAX_CLICKS est calcule automatiquement depuis le total d'avis affiche sur la page
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        """
        Lit le nombre total d'avis affiche sur la page (ex: "737 reviews").
        Essaie plusieurs selecteurs pour couvrir les deux layouts (page hotel / modale).
        Retourne un entier, ou None si non trouve.
        """
        selectors = [
            # Bouton "See all 737 reviews" sur la page principale
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            # Titre de la modale "Guest reviews" (pas de nombre ici)
            # Compteur dans la section avis (ex: "737 reviews")
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        # Fallback : chercher n'importe quel texte "N reviews" dans la page
        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:   # ignorer les "0 reviews helpful" etc.
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        """
        Calcule le nombre de clics necessaires :
            max_clicks = ceil(total_reviews / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        """
        total = self._get_total_reviews_count()
        if total is None:
            return 100  # valeur par defaut conservative

        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        """
        Deux layouts observes :
        1. Page hotel normale -> bouton data-stid='reviews-link' ouvre une modale
        2. URL avec ?pwaDialog=product-reviews -> modale deja ouverte OU bouton
           uitk-button-secondary 'See all N reviews' present en bas de page
        """
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        # Methode 1 : conteneurs data-stid explicites
        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        # Methode 2 (fallback) : articles avec h3 "out of 10"
        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""

        # Priorite 1 : div.uitk-expando-peek-inner (layout confirme — texte tronque)
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        # Priorite 2 : span[itemprop="description"] (layout alternatif)
        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        # Priorite 3 : heuristique — plus long div.uitk-type-300 different de trip_type/pub_date
        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        # Fermer popup eventuel
        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        # Calculer max_clicks AVANT d'ouvrir la modale (le compteur est visible sur la page)
        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Hotel Ibis Tanger City Center
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   14 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +10 | Total : 30
      'More reviews' clique
   Click 3 | +10 | Total : 40
      'More reviews' clique
   Click 4 | +10 | Total : 50
      'More reviews' clique
   Click 5 | +10 | Total : 60
      'More reviews' clique
   Click 6 | +10 | Total : 70
      'More reviews' clique
   Click 7 | +10 | Total : 80
      'More reviews' clique
   Click 8 | +10 | Total : 90
      'More reviews' clique
   Click 9 | +10 | Total : 100
      'More reviews' clique
   Click 10 | +10 | Total : 110
      'More reviews' clique
   Click 11 | +9 | Total : 119
      'More reviews' clique
   Click 12 | +10 | Total : 129
      'More reviews' clique
   Click 13 | +10 | Total : 139
      'More reviews' clique
   Click 1

In [2]:
df.head()

,source,hotel_name,hotel_city,reviewer_name,rating,review_text,trip_type,publication_date,stay_date,language
0,expedia.com,Hotel Ibis Tanger City Center,Tanger,MOHEB Mohsen,6,good for one or two days trip,Traveled with partner,"Feb 20, 2026",2 nights in Feb 2026,english
1,expedia.com,Hotel Ibis Tanger City Center,Tanger,Ratawan,2,Convenience next to train station and airport ...,Traveled with partner,"Mar 8, 2026",1 night in Mar 2026,english
2,expedia.com,Hotel Ibis Tanger City Center,Tanger,Ryan,10,Good location and great staff,"Dec 25, 2025",,4 nights in Oct 2025,english
3,expedia.com,Hotel Ibis Tanger City Center,Tanger,Hector,8,Nice hotel close to train station and city beach,Solo traveler,"Jan 5, 2026",1 night in Dec 2025,english
4,expedia.com,Hotel Ibis Tanger City Center,Tanger,Kala,6,Properly location was great and lobby was good...,Traveled with family and young children,"Mar 27, 2026",2 nights in Mar 2026,english


In [5]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Casablanca-Hotels-Ibis-Mohammedia.h19403556.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis  Mohammedia"
HOTEL_CITY = "Mohammedia"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis  Mohammedia
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   19 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +20 | Total : 40
      'More reviews' clique
   Click 3 | +30 | Total : 70
      'More reviews' clique
   Click 4 | +0 | Total : 70
      'More reviews' clique
   Click 5 | +0 | Total : 70
      'More reviews' clique
   Click 6 | +0 | Total : 70
      'More reviews' clique
   Click 7 | +20 | Total : 90
      'More reviews' clique
   Click 8 | +10 | Total : 100
      'More reviews' clique
   Click 9 | +10 | Total : 110
      'More reviews' clique
   Click 10 | +8 | Total : 118
      'More reviews' clique
   Click 11 | +29 | Total : 147
      Retry — scroll complet + attente 2s...
      Bouton 'More reviews' introuvable — fin des avis.
   Fin naturelle — plus de bouton 'More reviews'.

147 avis

In [6]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL        = "https://www.expedia.com/Casablanca-Hotels-Ibis-Abdelmoumen.h45709254.Hotel-Information?pwaDialog=product-reviews"
HOTEL_NAME = "Ibis  Abdelmoumen Casa Center"
HOTEL_CITY = "Casablanca"
REVIEWS_PER_CLICK = 10
SAFETY_MARGIN     = 10
# ==================================================

def human_sleep(a=0.4, b=1.0):
    time.sleep(random.uniform(a, b))


class ExpediaScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en-US')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(60)
        self.wait      = WebDriverWait(self.driver, 15)
        self.wait_long = WebDriverWait(self.driver, 25)
        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()
        self._lang_map = {
            Language.FRENCH:     "french",
            Language.ENGLISH:    "english",
            Language.GERMAN:     "german",
            Language.SPANISH:    "spanish",
            Language.ITALIAN:    "italian",
            Language.PORTUGUESE: "portuguese",
            Language.DUTCH:      "dutch",
            Language.RUSSIAN:    "russian",
            Language.ARABIC:     "arabic",
            Language.CHINESE:    "chinese",
        }

    # ------------------------------------------------------------------ helpers
    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    # ------------------------------------------------------------------ auto max_clicks
    def _get_total_reviews_count(self):
        selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in selectors:
            try:
                els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    text = (el.get_attribute("aria-label") or el.text or "").strip()
                    m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                    if m:
                        count = int(m.group(1).replace(",", ""))
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
            except Exception:
                pass

        try:
            els = self.driver.find_elements(By.XPATH, "//*[contains(text(),'reviews')]")
            for el in els:
                text = el.text.strip()
                m = re.search(r'(\d[\d,]*)\s*reviews?', text, re.I)
                if m:
                    count = int(m.group(1).replace(",", ""))
                    if count > 5:
                        print(f"   Total avis detecte : {count}  (depuis '{text}')")
                        return count
        except Exception:
            pass

        print("   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut")
        return None

    def _compute_max_clicks(self):
        total = self._get_total_reviews_count()
        if total is None:
            return 100
        import math
        max_clicks = math.ceil(total / REVIEWS_PER_CLICK) + SAFETY_MARGIN
        print(f"   max_clicks calcule : ceil({total}/{REVIEWS_PER_CLICK}) + {SAFETY_MARGIN} = {max_clicks}")
        return max_clicks

    # ------------------------------------------------------------------ captcha
    def _wait_for_human_validation(self):
        print("\n" + "="*60)
        print("CAPTCHA ou page bloquee !")
        print("Resous le captcha dans Chrome si necessaire")
        print("Assure-toi que les avis sont visibles")
        print("Appuie sur ENTREE ici quand c'est pret...")
        print("="*60)
        input()
        print("   Verification des avis...")
        for attempt in range(10):
            cards = self._get_review_cards()
            if cards:
                print(f"   {len(cards)} avis visibles — on continue !")
                return True
            print(f"   Attente... ({attempt+1}/10)")
            time.sleep(1)
        input("   Appuie sur ENTREE quand les avis sont visibles...")
        return True

    # ------------------------------------------------------------------ open reviews
    def _click_see_all_reviews(self):
        print("   Verification ouverture modale reviews...")
        human_sleep(1.5, 2.5)
        if self._get_review_cards():
            print("   Avis deja visibles")
            return True

        print("   Clic sur le bouton reviews...")
        css_selectors = [
            "button[data-stid='reviews-link']",
            "button[aria-label*='reviews']",
            "button[aria-label*='Reviews']",
            "button.uitk-button.uitk-button-secondary",
        ]
        for sel in css_selectors:
            try:
                btns = self.driver.find_elements(By.CSS_SELECTOR, sel)
                for btn in btns:
                    label = (btn.get_attribute("aria-label") or btn.text or "").strip().lower()
                    if "review" not in label:
                        continue
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    human_sleep(0.4, 0.8)
                    btn.click()
                    print(f"   Bouton clique : '{label}'")
                    human_sleep(2.0, 3.0)
                    if self._get_review_cards():
                        return True
            except Exception:
                pass

        for xpath in [
            "//button[contains(@aria-label,'reviews')]",
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'reviews')]",
        ]:
            try:
                btn = self.driver.find_element(By.XPATH, xpath)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.4, 0.8)
                btn.click()
                human_sleep(2.0, 3.0)
                print("   Bouton reviews clique (XPath)")
                return True
            except Exception:
                pass

        print("   Bouton 'See all reviews' introuvable")
        return False

    # ------------------------------------------------------------------ snapshot
    def _get_snapshot(self):
        try:
            cards = self._get_review_cards()
            if cards:
                last = cards[-1]
                name_els = last.find_elements(By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7")
                return f"{len(cards)}__{name_els[0].text if name_els else ''}"
        except Exception:
            pass
        return ""

    # ------------------------------------------------------------------ load more
    def _click_more_reviews(self):
        selectors = [
            (By.CSS_SELECTOR, "button#load-more-reviews"),
            (By.XPATH,        "//button[normalize-space(text())='More reviews']"),
            (By.XPATH,        "//button[contains(@class,'uitk-button-secondary') and contains(text(),'More')]"),
        ]
        for by, sel in selectors:
            try:
                btn = self.wait_long.until(EC.element_to_be_clickable((by, sel)))
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.6, 1.2)
                btn.click()
                print("      'More reviews' clique")
                return True
            except Exception:
                pass

        print("      Retry — scroll complet + attente 2s...")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        for by, sel in selectors:
            try:
                btn = self.driver.find_element(by, sel)
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                human_sleep(0.5, 1.0)
                btn.click()
                print("      'More reviews' clique (retry)")
                return True
            except Exception:
                pass

        print("      Bouton 'More reviews' introuvable — fin des avis.")
        return False

    # ------------------------------------------------------------------ expand
    def _expand_reviews(self):
        try:
            buttons = self.driver.find_elements(
                By.XPATH,
                "//button[contains(text(),'See more') or contains(text(),'Read more')]"
            )
            for btn in buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    btn.click()
                    time.sleep(0.1)
                except Exception:
                    pass
        except Exception:
            pass

    # ------------------------------------------------------------------ review cards
    def _get_review_cards(self):
        cards = []
        seen_ids = set()

        try:
            items = self.driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-stid='product-reviews-list-item']"
            )
            for item in items:
                arts = item.find_elements(By.CSS_SELECTOR, "article")
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
        except Exception:
            pass

        if not cards:
            try:
                arts = self.driver.find_elements(
                    By.XPATH, "//article[.//h3[contains(@aria-label,'out of 10')]]"
                )
                for art in arts:
                    try:
                        eid = art.id
                    except Exception:
                        eid = None
                    if eid and eid in seen_ids:
                        continue
                    if eid:
                        seen_ids.add(eid)
                    cards.append(art)
            except Exception:
                pass

        return cards

    # ------------------------------------------------------------------ parse one card
    def _parse_card(self, card):
        # ---- Rating ----
        rating = ""
        try:
            h3 = card.find_element(By.XPATH, ".//h3[contains(@aria-label,'out of 10')]")
            aria = h3.get_attribute("aria-label") or ""
            m = re.search(r'(\d+(?:\.\d+)?)\s*out of 10', aria)
            if m:
                rating = m.group(1)
        except Exception:
            pass

        # ---- Reviewer name ----
        name = ""
        try:
            name = card.find_element(
                By.CSS_SELECTOR, "h4.uitk-heading.uitk-heading-7"
            ).text.strip()
        except Exception:
            pass

        # ---- Trip type & publication date ----
        trip_type = ""
        pub_date  = ""
        try:
            grid_item = card.find_element(By.CSS_SELECTOR, "div.uitk-layout-grid-item")
            type300   = grid_item.find_elements(
                By.CSS_SELECTOR, "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
            )
            if len(type300) >= 1:
                trip_type = type300[0].text.strip()
            if len(type300) >= 2:
                pub_date  = type300[1].text.strip()
        except Exception:
            pass

        # ---- Stay date ----
        stay_raw = ""
        try:
            stay_raw = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-text.uitk-type-200.uitk-text-standard-theme.uitk-layout-flex-item"
            ).text.strip()
        except Exception:
            pass
        stay_date = re.sub(r'^Stayed\s*', '', stay_raw).strip()

        # ---- Review text ----
        text = ""
        try:
            text = card.find_element(
                By.CSS_SELECTOR,
                "div.uitk-expando-peek-inner div.uitk-text.uitk-type-300"
            ).text.strip()
        except Exception:
            pass

        if not text:
            try:
                text = card.find_element(
                    By.CSS_SELECTOR, "span[itemprop='description']"
                ).text.strip()
            except Exception:
                pass

        if not text:
            try:
                candidates = card.find_elements(
                    By.CSS_SELECTOR,
                    "div.uitk-text.uitk-type-300.uitk-text-standard-theme"
                )
                for div in candidates:
                    t = div.text.strip()
                    if t and t != trip_type and t != pub_date and len(t) > 30:
                        text = t
                        break
            except Exception:
                pass

        return {
            "source":           "expedia.com",
            "hotel_name":       HOTEL_NAME,
            "hotel_city":       HOTEL_CITY,
            "reviewer_name":    name,
            "rating":           rating,
            "review_text":      text,
            "trip_type":        trip_type,
            "publication_date": pub_date,
            "stay_date":        stay_date,
            "language":         self.detect_language(text),
        }

    # ------------------------------------------------------------------ main scrape
    def scrape(self, url):
        print(f"Scraping Expedia : {HOTEL_NAME}")
        self.driver.get(url)
        human_sleep(2.5, 4.0)

        for sel in [
            "button[data-stid='modal-close']",
            "button[aria-label='Close']",
            "button[data-testid='close-button']",
            "button[aria-label='Close guest reviews dialog']",
        ]:
            try:
                self.driver.find_element(By.CSS_SELECTOR, sel).click()
                human_sleep(0.4, 0.8)
                print("   Popup ferme")
                break
            except Exception:
                pass

        max_clicks = self._compute_max_clicks()

        if not self._click_see_all_reviews():
            self._wait_for_human_validation()
            self._click_see_all_reviews()

        human_sleep(1.5, 2.5)

        if not self._get_review_cards():
            self._wait_for_human_validation()

        all_reviews = []
        seen        = set()
        click_count = 0

        print(f"   Collecte des avis (max {max_clicks} clics)...")

        while True:
            self._expand_reviews()
            cards = self._get_review_cards()

            if not cards and click_count > 0:
                print("\n   Avis disparus — captcha reapparu ?")
                self._wait_for_human_validation()
                cards = self._get_review_cards()

            new_count = 0
            for card in cards:
                review = self._parse_card(card)

                if not review.get("reviewer_name") and not review.get("review_text"):
                    continue

                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("review_text", "")[:50],
                )
                if key in seen:
                    continue
                seen.add(key)
                all_reviews.append(review)
                new_count += 1

            print(f"   Click {click_count} | +{new_count} | Total : {len(all_reviews)}")

            if click_count >= max_clicks:
                print(f"   Limite {max_clicks} clics atteinte.")
                break

            snap_before = self._get_snapshot()
            if not self._click_more_reviews():
                print("   Fin naturelle — plus de bouton 'More reviews'.")
                break

            for _ in range(20):
                time.sleep(0.3)
                if self._get_snapshot() != snap_before:
                    break

            human_sleep(0.8, 1.5)
            click_count += 1

        self.driver.quit()
        print(f"\n{len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = ExpediaScraper()
    df = scraper.scrape(URL)

    if not df.empty:
        filename = f"avis_Expedia_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSauvegarde : {filename}  ({len(df)} avis)")
        print("\nApercu :")
        cols = [c for c in ["reviewer_name", "rating", "trip_type",
                             "stay_date", "publication_date", "language"]
                if c in df.columns]
        print(df[cols].head(10).to_string())
        print("\nAvis avec/sans commentaire :")
        has_text = df["review_text"].str.strip().ne("").sum()
        no_text  = df["review_text"].str.strip().eq("").sum()
        print(f"   Avec commentaire : {has_text}")
        print(f"   Sans commentaire : {no_text}")
    else:
        print("Aucun avis extrait.")

Scraping Expedia : Ibis  Abdelmoumen Casa Center
   Impossible de detecter le total d'avis — utilisation de MAX_CLICKS=100 par defaut
   Verification ouverture modale reviews...
   Clic sur le bouton reviews...
   Bouton 'See all reviews' introuvable

CAPTCHA ou page bloquee !
Resous le captcha dans Chrome si necessaire
Assure-toi que les avis sont visibles
Appuie sur ENTREE ici quand c'est pret...


   Verification des avis...
   16 avis visibles — on continue !
   Verification ouverture modale reviews...
   Avis deja visibles
   Collecte des avis (max 100 clics)...
   Click 0 | +10 | Total : 10
      'More reviews' clique
   Click 1 | +10 | Total : 20
      'More reviews' clique
   Click 2 | +30 | Total : 50
      'More reviews' clique
   Click 3 | +36 | Total : 86
      'More reviews' clique
   Click 4 | +19 | Total : 105
      'More reviews' clique
   Click 5 | +0 | Total : 105
      'More reviews' clique
   Click 6 | +0 | Total : 105
      'More reviews' clique
   Click 7 | +0 | Total : 105
      'More reviews' clique
   Click 8 | +24 | Total : 129
      Retry — scroll complet + attente 2s...
      Bouton 'More reviews' introuvable — fin des avis.
   Fin naturelle — plus de bouton 'More reviews'.

129 avis extraits au total

Sauvegarde : avis_Expedia_Ibis__Abdelmoumen_Casa_Center.csv  (129 avis)

Apercu :
  reviewer_name rating                                                  

In [10]:
import pandas as pd
import glob

files = glob.glob("./avis_Expedia_*.csv")
Avis_expedia_total = pd.concat(
    [pd.read_csv(f) for f in files], ignore_index=True
)

print(f"Shape : {Avis_expedia_total.shape}")
Avis_expedia_total.to_csv("Avis_expedia_total.csv", index=False)

Shape : (5049, 10)


In [11]:
Avis_expedia_total.head()

,source,hotel_name,hotel_city,reviewer_name,rating,review_text,trip_type,publication_date,stay_date,language
0,expedia.com,Ibis Oujda,Oujda,Jawad,6,Clean room,Traveled with family,"Apr 2, 2026",2 nights in Mar 2026,english
1,expedia.com,Ibis Oujda,Oujda,Dyan,10,Clean spacious rooms. All staff are thoughtful...,Traveled with partner,"Jun 23, 2023",3 nights in Jun 2023,english
2,expedia.com,Ibis Oujda,Oujda,John,10,Nice room with a great view! Ibis Oujda is ver...,Solo traveler,"Sep 1, 2025",3 nights in Aug 2025,english
3,expedia.com,Ibis Oujda,Oujda,Gordon,8,The food was excellent and there is a nice pat...,Solo traveler,"Jun 10, 2024",2 nights in Jun 2024,english
4,expedia.com,Ibis Oujda,Oujda,Mohamed El,2,No room available for my family and no refund ...,Traveled with family and young children,"Oct 21, 2025",3 nights in Oct 2025,english


In [13]:
Avis_expedia_total = Avis_expedia_total.dropna(subset=["review_text"])
Avis_expedia_total = Avis_expedia_total[Avis_expedia_total["review_text"].str.strip() != ""]

print(f"Shape après nettoyage : {Avis_expedia_total.shape}")
Avis_expedia_total.to_csv("Avis_expedia_total.csv", index=False)

Shape après nettoyage : (3938, 10)


In [14]:
import re

def fix_date_shift(row):
    # Pattern de date : "Mon DD, YYYY"
    date_pattern = r'^[A-Z][a-z]{2} \d{1,2}, \d{4}$'
    
    if pd.isna(row['publication_date']) and not pd.isna(row['trip_type']):
        if re.match(date_pattern, str(row['trip_type']).strip()):
            row['publication_date'] = row['trip_type']
            row['trip_type'] = None
    return row

Avis_expedia_total = Avis_expedia_total.apply(fix_date_shift, axis=1)

# Vérification
print(Avis_expedia_total[['trip_type', 'publication_date']].head(20))

                                  trip_type publication_date
0                      Traveled with family      Apr 2, 2026
1                     Traveled with partner     Jun 23, 2023
2                             Solo traveler      Sep 1, 2025
3                             Solo traveler     Jun 10, 2024
4   Traveled with family and young children     Oct 21, 2025
5                       Traveled with group     Oct 25, 2024
6                             Solo traveler      Jan 9, 2024
7                             Solo traveler      May 1, 2025
8                     Traveled with partner     Sep 17, 2025
9                      Traveled with family     Sep 11, 2025
10                    Traveled with partner      Sep 8, 2025
11                     Traveled with family     May 20, 2025
12                    Traveled with partner      Jan 5, 2025
13                            Solo traveler     Aug 28, 2023
14                                      NaN      Aug 1, 2023
15                      

In [15]:
# Sauvegarde
Avis_expedia_total.to_csv("Avis_expedia_total.csv", index=False)
print("Sauvegardé ✓")

Sauvegardé ✓


In [16]:
Avis_expedia_total.head()

,source,hotel_name,hotel_city,reviewer_name,rating,review_text,trip_type,publication_date,stay_date,language
0,expedia.com,Ibis Oujda,Oujda,Jawad,6,Clean room,Traveled with family,"Apr 2, 2026",2 nights in Mar 2026,english
1,expedia.com,Ibis Oujda,Oujda,Dyan,10,Clean spacious rooms. All staff are thoughtful...,Traveled with partner,"Jun 23, 2023",3 nights in Jun 2023,english
2,expedia.com,Ibis Oujda,Oujda,John,10,Nice room with a great view! Ibis Oujda is ver...,Solo traveler,"Sep 1, 2025",3 nights in Aug 2025,english
3,expedia.com,Ibis Oujda,Oujda,Gordon,8,The food was excellent and there is a nice pat...,Solo traveler,"Jun 10, 2024",2 nights in Jun 2024,english
4,expedia.com,Ibis Oujda,Oujda,Mohamed El,2,No room available for my family and no refund ...,Traveled with family and young children,"Oct 21, 2025",3 nights in Oct 2025,english


In [17]:
Avis_expedia_total.columns

Index(['source', 'hotel_name', 'hotel_city', 'reviewer_name', 'rating',
       'review_text', 'trip_type', 'publication_date', 'stay_date',
       'language'],
      dtype='str')